In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn import metrics

In [2]:
data = pd.read_csv('data/data_DFT_MF.csv')
y = data[['Yield']]
X = data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])
pc_cols  = [c for c in X.columns if c.startswith("PC_MF_")]
alc_cols = [c for c in X.columns if c.startswith("Alc_MF_")]
print(f"PC columns: {len(pc_cols)}")
print(f"Alc columns: {len(alc_cols)}")

pc_pipeline = Pipeline([("pca", PCA(random_state=0))])
alc_pipeline = Pipeline([("pca", PCA(random_state=0))])
preprocessor = ColumnTransformer([("pc",  pc_pipeline,  pc_cols),("alc", alc_pipeline, alc_cols)], remainder="passthrough")

pipe = Pipeline([("preprocess", preprocessor),
                 ("model", HistGradientBoostingRegressor(random_state=0,max_leaf_nodes=5,max_bins=30))])

param_grid = {"model__min_samples_leaf": [2, 3, 5],
              "model__max_depth": [4, 6],
              "model__l2_regularization": [0, 0.1, 1],
              "preprocess__pc__pca__n_components": [5, 6, 7],
              "preprocess__alc__pca__n_components": [5, 6, 7]}

r2_score_list = []
rmse_score_list = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=i)
    reg = GridSearchCV(pipe, param_grid=param_grid, cv=5, n_jobs=12)
    reg.fit(X_train, y_train["Yield"])
    best = reg.best_estimator_
    y_pred_test = best.predict(X_test)
    r2 = metrics.r2_score(y_test["Yield"], y_pred_test)
    rmse = metrics.root_mean_squared_error(y_test["Yield"], y_pred_test)
    alc_pcs = best.named_steps["preprocess"].named_transformers_["alc"].named_steps["pca"].n_components_
    pc_pcs  = best.named_steps["preprocess"].named_transformers_["pc"].named_steps["pca"].n_components_
    print(f'Run{i} R2: {r2:.2f}, RMSE: {rmse:.2f} | Alc_PCs: {alc_pcs}, PC_PCs: {pc_pcs}')
    r2_score_list.append(r2)
    rmse_score_list.append(rmse)
    #print(best.named_steps["preprocess"].get_feature_names_out())
print('==========(Result)==========')
print(f'Mean R2:  {np.mean(r2_score_list):.3f}')
print(f'SD R2:    {np.std(r2_score_list):.3f}')
print(f'Mean RMSE:{np.mean(rmse_score_list):.3f}')
print(f'SD RMSE:  {np.std(rmse_score_list):.3f}')

PC columns: 102
Alc columns: 108
Run0 R2: 0.71, RMSE: 14.44 | Alc_PCs: 6, PC_PCs: 7
Run1 R2: 0.69, RMSE: 11.70 | Alc_PCs: 5, PC_PCs: 7
Run2 R2: 0.88, RMSE: 6.55 | Alc_PCs: 6, PC_PCs: 5
Run3 R2: 0.76, RMSE: 13.10 | Alc_PCs: 7, PC_PCs: 7
Run4 R2: 0.74, RMSE: 12.48 | Alc_PCs: 5, PC_PCs: 6
Run5 R2: 0.74, RMSE: 10.69 | Alc_PCs: 6, PC_PCs: 7
Run6 R2: 0.79, RMSE: 9.13 | Alc_PCs: 7, PC_PCs: 7
Run7 R2: 0.74, RMSE: 10.44 | Alc_PCs: 6, PC_PCs: 7
Run8 R2: 0.65, RMSE: 13.13 | Alc_PCs: 5, PC_PCs: 6
Run9 R2: 0.64, RMSE: 11.91 | Alc_PCs: 5, PC_PCs: 7
==========(Result)==========
Mean R2:  0.734
SD R2:    0.064
Mean RMSE:11.358
SD RMSE:  2.161


In [3]:
data = pd.read_csv('data/data_DFT_RDKit.csv')
y = data[['Yield']]
X = data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])

pc_cols  = [c for c in X.columns if c.startswith("RDKit_PC_")]
alc_cols = [c for c in X.columns if c.startswith("RDKit_Alc_")]
print(f"PC columns: {len(pc_cols)}")
print(f"Alc columns: {len(alc_cols)}")

pc_pipeline = Pipeline([("scaler", StandardScaler()),
                        ("pca", PCA(random_state=0))])
alc_pipeline = Pipeline([("scaler", StandardScaler()),
                         ("pca", PCA(random_state=0))])
preprocessor = ColumnTransformer([("pc",  pc_pipeline,  pc_cols),("alc", alc_pipeline, alc_cols)], remainder="passthrough")

pipe = Pipeline([("preprocess", preprocessor),
                 ("model", HistGradientBoostingRegressor(random_state=0,max_leaf_nodes=5,max_bins=30))])

param_grid = {"model__min_samples_leaf": [2, 3, 5],
              "model__max_depth": [4, 6],
              "model__l2_regularization": [0, 0.1, 1],
              "preprocess__pc__pca__n_components": [5, 6, 7],
              "preprocess__alc__pca__n_components": [5, 6, 7]}

r2_score_list = []
rmse_score_list = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=i)
    reg = GridSearchCV(pipe, param_grid=param_grid, cv=5, n_jobs=12)
    reg.fit(X_train, y_train["Yield"])
    best = reg.best_estimator_
    y_pred_test = best.predict(X_test)
    r2 = metrics.r2_score(y_test["Yield"], y_pred_test)
    rmse = metrics.root_mean_squared_error(y_test["Yield"], y_pred_test)
    alc_pcs = best.named_steps["preprocess"].named_transformers_["alc"].named_steps["pca"].n_components_
    pc_pcs  = best.named_steps["preprocess"].named_transformers_["pc"].named_steps["pca"].n_components_
    print(f'Run{i} R2: {r2:.2f}, RMSE: {rmse:.2f} | Alc_PCs: {alc_pcs}, PC_PCs: {pc_pcs}')
    r2_score_list.append(r2)
    rmse_score_list.append(rmse)
print('==========(Result)==========')
print(f'Mean R2:  {np.mean(r2_score_list):.3f}')
print(f'SD R2:    {np.std(r2_score_list):.3f}')
print(f'Mean RMSE:{np.mean(rmse_score_list):.3f}')
print(f'SD RMSE:  {np.std(rmse_score_list):.3f}')

PC columns: 129
Alc columns: 118
Run0 R2: 0.67, RMSE: 15.40 | Alc_PCs: 7, PC_PCs: 6
Run1 R2: 0.69, RMSE: 11.55 | Alc_PCs: 5, PC_PCs: 5
Run2 R2: 0.81, RMSE: 8.08 | Alc_PCs: 6, PC_PCs: 5
Run3 R2: 0.67, RMSE: 15.28 | Alc_PCs: 7, PC_PCs: 6
Run4 R2: 0.77, RMSE: 11.86 | Alc_PCs: 5, PC_PCs: 6
Run5 R2: 0.73, RMSE: 10.88 | Alc_PCs: 5, PC_PCs: 5
Run6 R2: 0.71, RMSE: 10.60 | Alc_PCs: 7, PC_PCs: 6
Run7 R2: 0.63, RMSE: 12.58 | Alc_PCs: 5, PC_PCs: 6
Run8 R2: 0.80, RMSE: 10.02 | Alc_PCs: 5, PC_PCs: 6
Run9 R2: 0.64, RMSE: 12.01 | Alc_PCs: 6, PC_PCs: 6
==========(Result)==========
Mean R2:  0.713
SD R2:    0.061
Mean RMSE:11.828
SD RMSE:  2.124


In [4]:
data = pd.read_csv('data/data_DFT_mordred.csv')
y = data[['Yield']]
X = data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])

pc_cols  = [c for c in X.columns if c.startswith("mordred_PC_")]
alc_cols = [c for c in X.columns if c.startswith("mordred_Alc_")]
print(f"PC columns: {len(pc_cols)}")
print(f"Alc columns: {len(alc_cols)}")

pc_pipeline = Pipeline([("scaler", StandardScaler()),
                        ("pca", PCA(random_state=0))])
alc_pipeline = Pipeline([("scaler", StandardScaler()),
                         ("pca", PCA(random_state=0))])
preprocessor = ColumnTransformer([("pc",  pc_pipeline,  pc_cols),("alc", alc_pipeline, alc_cols)], remainder="passthrough")

pipe = Pipeline([("preprocess", preprocessor),
                 ("model", HistGradientBoostingRegressor(random_state=0,max_leaf_nodes=5,max_bins=30))])

param_grid = {"model__min_samples_leaf": [2, 3, 5],
              "model__max_depth": [4, 6],
              "model__l2_regularization": [0, 0.1, 1],
              "preprocess__pc__pca__n_components": [5, 6, 7],
              "preprocess__alc__pca__n_components": [5, 6, 7]}

r2_score_list = []
rmse_score_list = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=i)
    reg = GridSearchCV(pipe, param_grid=param_grid, cv=5, n_jobs=12)
    reg.fit(X_train, y_train["Yield"])
    best = reg.best_estimator_
    y_pred_test = best.predict(X_test)
    r2 = metrics.r2_score(y_test["Yield"], y_pred_test)
    rmse = metrics.root_mean_squared_error(y_test["Yield"], y_pred_test)
    alc_pcs = best.named_steps["preprocess"].named_transformers_["alc"].named_steps["pca"].n_components_
    pc_pcs  = best.named_steps["preprocess"].named_transformers_["pc"].named_steps["pca"].n_components_
    print(f'Run{i} R2: {r2:.2f}, RMSE: {rmse:.2f} | Alc_PCs: {alc_pcs}, PC_PCs: {pc_pcs}')
    r2_score_list.append(r2)
    rmse_score_list.append(rmse)
print('==========(Result)==========')
print(f'Mean R2:  {np.mean(r2_score_list):.3f}')
print(f'SD R2:    {np.std(r2_score_list):.3f}')
print(f'Mean RMSE:{np.mean(rmse_score_list):.3f}')
print(f'SD RMSE:  {np.std(rmse_score_list):.3f}')

PC columns: 1153
Alc columns: 1145
Run0 R2: 0.73, RMSE: 13.94 | Alc_PCs: 7, PC_PCs: 7
Run1 R2: 0.78, RMSE: 9.78 | Alc_PCs: 7, PC_PCs: 6
Run2 R2: 0.76, RMSE: 9.05 | Alc_PCs: 7, PC_PCs: 5
Run3 R2: 0.79, RMSE: 12.20 | Alc_PCs: 5, PC_PCs: 5
Run4 R2: 0.86, RMSE: 9.24 | Alc_PCs: 6, PC_PCs: 6
Run5 R2: 0.75, RMSE: 10.57 | Alc_PCs: 6, PC_PCs: 6
Run6 R2: 0.73, RMSE: 10.26 | Alc_PCs: 7, PC_PCs: 6
Run7 R2: 0.59, RMSE: 13.20 | Alc_PCs: 7, PC_PCs: 5
Run8 R2: 0.76, RMSE: 10.96 | Alc_PCs: 7, PC_PCs: 6
Run9 R2: 0.76, RMSE: 9.67 | Alc_PCs: 6, PC_PCs: 7
==========(Result)==========
Mean R2:  0.752
SD R2:    0.064
Mean RMSE:10.888
SD RMSE:  1.604
